# Training Dueling DQN Agents

This notebook trains the `DQNVec11Agent` and `DQNGridAgent` utilizing Dueling DQN architecture on vectorized environments. Dueling DQN separates state values from action advantages, drastically speeding up learning in collision-heavy environments like Snake.

In [9]:
import sys
import os

sys.path.append(os.path.abspath(".."))  # Ensure imports from core/agents work

from gymnasium.vector import SyncVectorEnv, AsyncVectorEnv
from gymnasium.wrappers import FrameStackObservation

from core.env.core import SnakeEnv
from core.env.types import ObserveType, RewardOptions
from agents.dqn.dqn_vec11 import DQNVec11Agent
from agents.dqn.dqn_grid import DQNGridAgent

## Environment Setup
Reward shaping is key. We apply heavy penalties for collisions, and give small dense rewards for making steps closer to the apple.

In [10]:
def make_env(env_id, obs_type, width=15, height=15, frame_stack=1):
    def _init():
        reward_options = RewardOptions(
            eats_apple=10.0,
            complete=100.0,
            penalty_step=-0.01,
            penalty_loop=-0.1,
            death_wall=-10.0,
            death_self=-10.0,
            shaping_closer=0.1,
            shaping_further=-0.1,
        )
        env = SnakeEnv(
            width=width,
            height=height,
            obs_type=obs_type,
            num_apples=1,
            num_obstacles=0,  # Let it learn basics before adding obstacles
            max_steps=2000,
            reward_options=reward_options,
        )
        if frame_stack > 1:
            env = FrameStackObservation(env, stack_size=frame_stack)
        return env

    return _init


num_envs = 16  # Parallelize experience gathering

## 1. Train Vec11 Agent
The 11-dimensional observation vector uses linear layers and usually learns extremely quickly.

In [11]:
envs_vec = SyncVectorEnv([make_env(i, ObserveType.VEC_11) for i in range(num_envs)])

agent_vec11 = DQNVec11Agent(learning_rate=3e-4, device="cuda", model_file="dqn_vec11_dueling.pth")

print("Training Vec11 Agent...")
stats_vec11 = agent_vec11.train(
    env=envs_vec,
    total_timesteps=500_000,
    batch_size=256,
    buffer_size=100_000,
    learning_starts=10_000,
    eps_init=1.0,
    eps_final=0.01,
    eps_decay=0.5,  # Decay over first 50% of total timesteps
    log_step=10_000,
)

Training Vec11 Agent...


Parallel Training: 100%|██████████| 500000/500000 [20:05<00:00, 414.85it/s, Avg Rwd (100)=211.31, Best=240.90, Eps=0.010]


## 2. Train Grid Agent (CNN)
The CNN agent uses a 3D grid observation (channels for snake, apples, obstacles). We use frame stacking (`frame_stack=2`) so the network can infer movement direction from consecutive frames. CNNs generally take longer to learn spatial features, so we increase `total_timesteps`.

In [8]:
envs_grid = AsyncVectorEnv(
    [make_env(i, ObserveType.FULL_GRID, frame_stack=2, width=20, height=20) for i in range(num_envs)]
)

agent_grid = DQNGridAgent(
    in_channels=4,
    grid_shape=(20, 20),
    frame_stack=2,
    learning_rate=1e-4,
    device="cuda",
    model_file="../artifacts/models/dqn_grid.pth",
)

print("Training Grid Agent...")
stats_grid = agent_grid.train(
    env=envs_grid,
    total_timesteps=500_000,
    batch_size=256,
    buffer_size=100_000,
    learning_starts=5000,
    train_freq=4,
    eps_init=1.0,
    eps_final=0.01,
    eps_decay=0.6,
    log_step=10_000,
)

Training Grid Agent...


Parallel Training: 100%|██████████| 500000/500000 [1:18:31<00:00, 106.12it/s, Avg Rwd (100)=66.88, Best=76.46, Eps=0.010] 
